# 03 · GHCNh weather → clean hourly UTC
Source: the GHCNh station files (`data/raw/ghcnh/GHCNh_{station}_por.psv`). The documentation is `reference/ghcnh_DOCUMENTATION.pdf`.

Units in the file are converted:
- wind_speed / wind_gust: m/s → kt
- visibility: km → statute miles (16.093 km = 10 sm, the reporting maximum)
- ceiling_height: m → ft (22,000 m is coded as unlimited)
- precipitation: mm, no conversion

**Hour convention:** row `hour_utc = t` describes the interval [t, t+1). Instantaneous variables use the *last* valid observation inside that hour. That's normally the routine METAR at about :51 (`FM-15`), or a later special report (`FM-16`). Synoptic `FM-12` rows are dropped, since they duplicate the METARs at coarser precision.

Output: `data/interim/weather_{APT}.parquet`

In [1]:
%run ../config.ipynb

ROOT = /Users/dylannegvesky/Documents/FSU Documents/Fall 2026/ProjDS/PDS_Final
study airports: ['EWR'] | BTS 2023-01 → latest | train ≤ 2025-12


In [2]:
QC_VARS = ["temperature", "dew_point_temperature", "sea_level_pressure", "wind_direction",
           "wind_speed", "wind_gust", "precipitation", "visibility", "ceiling_height", "snow_depth"]
WX_CODE_COLS = [f"pres_wx_{s}{i}" for s in ("MW", "AU", "AW") for i in (1, 2, 3)]
USECOLS = (["DATE", "temperature_Report_Type"] + QC_VARS
           + [f"{v}_Quality_Code" for v in QC_VARS] + WX_CODE_COLS)
FIRST_HOUR = (pd.Period(BTS_START, "M").start_time - pd.Timedelta(days=31)).tz_localize("UTC")


def load_station(station):
    path = GHCNH_DIR / f"GHCNh_{station}_por.psv"
    df = pd.read_csv(path, sep="|", usecols=USECOLS, low_memory=False,
                     dtype={f"{v}_Quality_Code": "string" for v in QC_VARS})
    df["time"] = pd.to_datetime(df["DATE"], utc=True)
    df = df[df["time"] >= FIRST_HOUR].sort_values("time")
    return df.rename(columns={"temperature_Report_Type": "report_type"})

## Quality control
A value is set to NaN when its quality code is suspect or erroneous (2, 3, 6, 7) or is a lowercase QC-check failure flag (docs §VI). A wind direction of 999 means variable or missing.

In [3]:
def apply_qc(df):
    df = df.copy()
    for v in QC_VARS:
        q = df[f"{v}_Quality_Code"].astype("string")
        bad = q.isin(GHCNH_BAD_QC) | q.str.fullmatch(r"[a-z]").fillna(False)
        df.loc[bad, v] = np.nan
    df.loc[df["wind_direction"] > 360, "wind_direction"] = np.nan
    return df


def clean_units(df):
    out = pd.DataFrame({"time": df["time"], "report_type": df["report_type"]})
    out["temp_c"] = df["temperature"]
    out["dewpoint_c"] = df["dew_point_temperature"]
    out["slp_hpa"] = df["sea_level_pressure"]
    out["wind_dir_deg"] = df["wind_direction"]
    out["wind_kt"] = df["wind_speed"] * MS_TO_KT
    out["gust_kt"] = df["wind_gust"] * MS_TO_KT
    out["vis_sm"] = (df["visibility"] * KM_TO_SM).clip(upper=10)
    cig_ft = df["ceiling_height"] * M_TO_FT
    out["ceiling_ft"] = cig_ft.where(df["ceiling_height"] < 22000, 25000).clip(upper=25000)  # unlimited → 25,000 ft
    out["precip_mm"] = df["precipitation"].where(df["report_type"] == "FM15")  # FM-15 holds the hourly total; FM-16 would double-count
    out["snow_depth_mm"] = df["snow_depth"]
    wx = df[WX_CODE_COLS].astype("string").fillna("")
    out["wx_codes"] = wx.agg(" ".join, axis=1).str.strip()
    return out

## Hourly aggregation and derived features
- **Flight category** (FAA): LIFR = ceiling < 500 ft or visibility < 1 sm; IFR < 1,000 ft / 3 sm; MVFR ≤ 3,000 ft / 5 sm; otherwise VFR. `imc` = IFR or LIFR.
- **Present-weather flags** are parsed from the METAR-style codes (`TS:95`, `SN:71`, `FZRA:66`, ...) across the manual and automated sensor columns.
- **Wind u/v**: the components of the wind's *from* direction. Calm wind gives u = v = 0.

In [4]:
WX_FLAGS = {
    "wx_ts": r"\b(?:VC)?TS",              # thunderstorm, including in the vicinity
    "wx_snow": r"\b[+-]?(?:SH)?(?:SN|SG|PL|GS)",
    "wx_freezing": r"FZ(?:RA|DZ|FG)|\b(?:56|57|66|67):",
    "wx_fog": r"\b(?:FG|BR)",
    "wx_rain": r"\b[+-]?(?:SH)?(?:RA|DZ)",
    "wx_heavy": r"\+",
}


def flight_category(cig, vis):
    cat = np.select([(cig < 500) | (vis < 1), (cig < 1000) | (vis < 3), (cig <= 3000) | (vis <= 5)],
                    [3, 2, 1], default=0)
    return np.where(cig.isna() & vis.isna(), np.nan, cat)   # 0 = VFR, 1 = MVFR, 2 = IFR, 3 = LIFR


def to_hourly(obs):
    obs = obs[obs["report_type"] != "FM12"].copy()
    obs["hour_utc"] = obs["time"].dt.floor("h")
    g = obs.groupby("hour_utc")
    inst = ["temp_c", "dewpoint_c", "slp_hpa", "wind_dir_deg", "wind_kt", "vis_sm", "ceiling_ft", "snow_depth_mm"]
    h = g[inst].last()                                   # last non-null value in the hour
    h["vis_min_sm"] = g["vis_sm"].min()
    h["ceiling_min_ft"] = g["ceiling_ft"].min()
    h["gust_kt"] = g["gust_kt"].max()
    h["wind_max_kt"] = g["wind_kt"].max()
    h["precip_mm"] = g["precip_mm"].sum(min_count=1)
    h["n_specials"] = g["report_type"].agg(lambda s: (s == "FM16").sum())
    h["n_obs"] = g.size()
    codes = g["wx_codes"].agg(" ".join)
    for col, pat in WX_FLAGS.items():
        h[col] = codes.str.contains(pat, regex=True).astype(int)
    return h


def finish(h):
    full = pd.date_range(h.index.min(), h.index.max(), freq="h", tz="UTC", name="hour_utc")
    h = h.reindex(full)
    h["wx_missing"] = h["n_obs"].isna().astype(int)
    h["n_obs"] = h["n_obs"].fillna(0)
    flags = [c for c in h if c.startswith("wx_") and c != "wx_missing"] + ["n_specials"]
    h[flags] = h[flags].fillna(0)
    h["snow_depth_mm"] = h["snow_depth_mm"].ffill(limit=24)
    cont = ["temp_c", "dewpoint_c", "slp_hpa", "wind_dir_deg", "wind_kt", "vis_sm", "ceiling_ft",
            "vis_min_sm", "ceiling_min_ft", "wind_max_kt"]
    h[cont] = h[cont].ffill(limit=WX_FFILL_LIMIT_H)
    h["precip_mm"] = h["precip_mm"].fillna(0).where(h["wx_missing"] == 0)
    h["gust_kt"] = h["gust_kt"].fillna(0)                 # no gust reported = no gust
    calm = h["wind_kt"] < CALM_WIND_KT
    h["wind_variable"] = (h["wind_dir_deg"].isna() & h["wind_kt"].notna() & ~calm).astype(int)
    rad = np.radians(h["wind_dir_deg"])
    h["wind_u"] = np.where(calm, 0, -h["wind_kt"] * np.sin(rad))
    h["wind_v"] = np.where(calm, 0, -h["wind_kt"] * np.cos(rad))
    h["flight_cat"] = flight_category(h["ceiling_ft"], h["vis_sm"])
    h["imc"] = (h["flight_cat"] >= 2).astype(float).where(h["flight_cat"].notna())
    return h.reset_index()

In [5]:
weather = {}
for apt, cfg in STUDY_AIRPORTS.items():
    raw = load_station(cfg["ghcnh_station"])
    hourly = finish(to_hourly(clean_units(apply_qc(raw))))
    hourly.insert(0, "airport", apt)
    weather[apt] = hourly
    hourly.to_parquet(INTERIM / f"weather_{apt}.parquet", index=False)
    print(f"{apt}: {len(raw):,} raw obs → {len(hourly):,} hours "
          f"({hourly['hour_utc'].min():%Y-%m-%d} → {hourly['hour_utc'].max():%Y-%m-%d})")

EWR: 122,851 raw obs → 34,672 hours (2022-12-01 → 2026-11-14)


## Checks

In [6]:
for apt, h in weather.items():
    yr = h.groupby(h["hour_utc"].dt.year).agg(hours=("hour_utc", "size"),
                                              pct_missing=("wx_missing", "mean"),
                                              pct_imc=("imc", "mean"),
                                              ts_hours=("wx_ts", "sum"),
                                              snow_hours=("wx_snow", "sum"))
    print(apt); display(yr.round(3))
    key_na = h[["wind_kt", "vis_sm", "ceiling_ft"]].isna().mean()
    print("share still missing after forward-fill:\n", key_na.round(4))
    assert h["hour_utc"].is_unique and h["hour_utc"].diff().dropna().eq(pd.Timedelta("1h")).all()
    assert yr["pct_missing"].max() < 0.02, "more than 2% of hours have no observation"

EWR


,hours,pct_missing,pct_imc,ts_hours,snow_hours
hour_utc,,,,,
2022,744,0.000,0.108,1.0,11.0
2023,8760,0.000,0.098,113.0,81.0
2024,8784,0.000,0.083,70.0,109.0
2025,8760,0.002,0.084,50.0,128.0
2026,7624,0.000,0.046,80.0,111.0


share still missing after forward-fill:
 wind_kt       0.0024
vis_sm        0.0027
ceiling_ft    0.0001
dtype: float64


Spot-check a summer thunderstorm afternoon against the raw rows.

In [7]:
h = weather["EWR"]
storm_day = h.loc[h["wx_ts"] == 1, "hour_utc"].dt.date.value_counts().idxmax()
print("day with the most thunderstorm hours:", storm_day)
h[h["hour_utc"].dt.date == storm_day][["hour_utc", "wind_dir_deg", "wind_kt", "gust_kt", "vis_sm",
                                       "ceiling_ft", "precip_mm", "flight_cat", "wx_ts", "wx_rain",
                                       "n_specials"]]

day with the most thunderstorm hours: 2023-06-26


,hour_utc,wind_dir_deg,wind_kt,gust_kt,vis_sm,ceiling_ft,precip_mm,flight_cat,wx_ts,wx_rain,n_specials
4968,2023-06-26 00:00:00+00:00,90.0,4.082072,0.000000,9.999724,25000.00000,0.0,0.0,0.0,0.0,1.0
4969,2023-06-26 01:00:00+00:00,110.0,4.082072,0.000000,9.999724,25000.00000,0.0,0.0,0.0,0.0,0.0
4970,2023-06-26 02:00:00+00:00,110.0,0.000000,0.000000,9.999724,25000.00000,0.0,0.0,0.0,0.0,0.0
4971,2023-06-26 03:00:00+00:00,160.0,0.000000,0.000000,8.999938,25000.00000,0.0,0.0,0.0,0.0,1.0
4972,2023-06-26 04:00:00+00:00,160.0,0.000000,0.000000,8.999938,25000.00000,0.0,0.0,0.0,0.0,0.0
4973,2023-06-26 05:00:00+00:00,150.0,2.915766,0.000000,9.999724,25000.00000,0.0,0.0,0.0,0.0,0.0
4974,2023-06-26 06:00:00+00:00,150.0,0.000000,0.000000,9.999724,13999.34428,0.0,0.0,0.0,0.0,0.0
4975,2023-06-26 07:00:00+00:00,80.0,2.915766,0.000000,9.999724,13999.34428,0.0,0.0,0.0,0.0,0.0
4976,2023-06-26 08:00:00+00:00,160.0,4.082072,0.000000,9.999724,15000.00048,0.0,0.0,0.0,0.0,0.0
4977,2023-06-26 09:00:00+00:00,80.0,2.915766,0.000000,9.999724,11000.65652,0.0,0.0,0.0,0.0,0.0
